# Pending Contract Auto-Cancel Audit

This notebook checks pending Ejari contracts for the configured Emirates IDs and calculates two candidate auto-cancel dates:

- `auto_cancel_1_day_date`: base date + 1 day
- `auto_cancel_5_day_date`: base date + 5 days

The base date is `OwnerContractSigningDate` from the contract details API. If that field is empty, the auto-cancel dates are left blank and `autocancel_base_source` is marked as missing.


## What This Notebook Does

1. Gets a fresh iPaaS bearer token and DLD token for each Emirates ID.
2. Fetches contract history from the DLD contract-history endpoint.
3. Filters contracts whose history status is `Pending`.
4. Calls the contract-details endpoint for each pending contract row value.
5. Writes CSV/JSON outputs after every Emirates ID, including failures and raw contract detail responses.

Run outputs are stored under `runs/pending_contract_autocancel_audit_<timestamp>/`.


In [ ]:

# @title Configuration and environment
import base64
import csv
import json
import os
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import requests

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    from notebook_operator_utils import choose_option
except Exception:
    choose_option = None

# Required order: 0512 first, then the other known users.
OWNER_EMIRATES_IDS = [
    "784195279540512",
    "784199515347708",
    "784199668638416",
    "784197265140323",
    "784198721116758",
]

REQUEST_TIMEOUT_SECONDS = 45
CURRENT_BEARER_TOKEN = None


def load_local_env(path=".env"):
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_local_env()


def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)


def require_secret(name):
    value = get_secret(name)
    if value is None or str(value).strip() == "":
        raise RuntimeError(f"Missing required secret/env var: {name}")
    return str(value).strip()


BASIC_AUTH = require_secret("BASIC_AUTH")
CONSUMER_ID = require_secret("CONSUMER_ID")
IDS_BASE_URL = require_secret("IDS_BASE_URL").rstrip("/")
DLD_BASE_URL = require_secret("DLD_BASE_URL").rstrip("/")
DLD_PROXY_URL = require_secret("DLD_PROXY_URL").rstrip("/")

try:
    REQUEST_TIMEOUT_SECONDS = int(get_secret("REQUEST_TIMEOUT_SECONDS", REQUEST_TIMEOUT_SECONDS))
except Exception:
    pass
REQUEST_TIMEOUT_SECONDS = min(max(REQUEST_TIMEOUT_SECONDS, 15), 60)

RUN_DIR = Path("runs") / f"pending_contract_autocancel_audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DETAIL_DIR = RUN_DIR / "contract_details"
HISTORY_DIR = RUN_DIR / "contract_history"
FAILURE_DIR = RUN_DIR / "failures"
for folder in (DETAIL_DIR, HISTORY_DIR, FAILURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Configured Emirates IDs, in processing order:")
for emirates_id in OWNER_EMIRATES_IDS:
    print("-", emirates_id)
print("Run folder:", RUN_DIR)
print("Request timeout seconds:", REQUEST_TIMEOUT_SECONDS)


In [ ]:

# @title Operator confirmation

def text_choice(prompt, options, default):
    lookup = {str(i): value for i, (value, _) in enumerate(options, start=1)}
    lookup.update({value.lower(): value for value, _ in options})
    print(prompt)
    for i, (value, label) in enumerate(options, start=1):
        marker = " [default]" if value == default else ""
        print(f"{i}. {label}{marker}")
    while True:
        answer = input("Select option: ").strip().lower()
        if not answer:
            return default
        selected = lookup.get(answer)
        if selected:
            return selected
        print("Please choose one of the listed options.")


def ask_choice(prompt, options, default, title="Notebook option"):
    if choose_option is not None:
        return choose_option(
            prompt,
            [{"value": value, "label": label} for value, label in options],
            default=default,
            title=title,
        )
    return text_choice(prompt, options, default)


master_selection = ask_choice(
    "Process all configured Emirates IDs?",
    [("all", "Process all"), ("confirm_each", "Ask before each Emirates ID")],
    default="all",
    title="Auto-cancel audit scope",
)
print("Selection:", master_selection)


In [ ]:

# @title Response parsing and output helpers

def response_payload(response):
    try:
        return response.json()
    except Exception:
        return {"_text": response.text}


def compact_payload(payload, max_chars=1200):
    try:
        text = json.dumps(payload, ensure_ascii=False)
    except Exception:
        text = str(payload)
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def validation_errors(payload):
    if not isinstance(payload, dict):
        return []
    errors = []
    for key in ("ValidationErrorsList", "ValidationErrors", "Errors", "ErrorList"):
        value = payload.get(key)
        if value is None and isinstance(payload.get("Response"), dict):
            value = payload["Response"].get(key)
        if isinstance(value, list):
            for item in value:
                if not isinstance(item, dict):
                    if item:
                        errors.append(str(item))
                    continue
                number = item.get("ErrorNumber")
                message = item.get("ErrorMessage")
                message_set = item.get("ErrorMessageSet") or {}
                english = message_set.get("EnglishName") if isinstance(message_set, dict) else None
                if number not in (None, 0, "0") or (message and str(message).strip()) or (english and "no errors" not in str(english).lower()):
                    errors.append(compact_payload(item, 500))
        elif isinstance(value, dict):
            errors.append(compact_payload(value, 500))
    return errors


def api_errors(payload):
    if not isinstance(payload, dict):
        return ["Response is not JSON object"]
    errors = validation_errors(payload)
    success_value = str(payload.get("success", "")).strip().lower()
    response_code = str(payload.get("ResponseCode", "")).strip().lower()
    if success_value == "false":
        errors.append(payload.get("message") or payload.get("Message") or "API success=false")
    if response_code and response_code not in {"success", "0"}:
        errors.append(payload.get("message") or payload.get("Message") or f"ResponseCode={payload.get('ResponseCode')}")
    if payload.get("ReasonCode") not in (None, "", 0, "0") and success_value != "true":
        errors.append(payload.get("message") or f"ReasonCode={payload.get('ReasonCode')}")
    return [str(error) for error in errors if str(error).strip()]


def safe_filename(value, max_len=120):
    text = str(value or "blank")
    text = re.sub(r"[^A-Za-z0-9._=-]+", "_", text)
    return text[:max_len].strip("._-") or "blank"


def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def write_csv(path, rows, fieldnames):
    with Path(path).open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def make_curl(method, url, headers=None, body=None):
    parts = ["curl --location", f"--request {method.upper()}", f"'{url}'"]
    for key, value in (headers or {}).items():
        parts.append(f"--header '{key}: {value}'")
    if body is not None:
        if not isinstance(body, str):
            body = json.dumps(body, ensure_ascii=False)
        parts.append(f"--data '{body}'")
    return " \\\n  ".join(parts)


def final_auth_headers(response, fallback_headers):
    final_headers = dict(fallback_headers or {})
    sent_headers = getattr(getattr(response, "request", None), "headers", {}) or {}
    for sent_key, sent_value in sent_headers.items():
        if str(sent_key).lower() in {"authorization", "token"}:
            existing_key = next((key for key in final_headers if str(key).lower() == str(sent_key).lower()), sent_key)
            final_headers[existing_key] = sent_value
    return final_headers


In [ ]:

# @title Authentication and API functions

def get_bearer_token():
    global CURRENT_BEARER_TOKEN
    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    response = requests.post(IDS_BASE_URL, headers=headers, data={}, timeout=REQUEST_TIMEOUT_SECONDS)
    payload = response_payload(response)
    if response.status_code >= 400:
        raise RuntimeError(f"iPaaS token failed HTTP {response.status_code}: {compact_payload(payload)}")
    token = payload.get("id_token") if isinstance(payload, dict) else None
    if not token:
        raise RuntimeError(f"iPaaS token response has no id_token: {compact_payload(payload)}")
    CURRENT_BEARER_TOKEN = token
    return token


def get_active_bearer(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN


def request_with_bearer(method, url, headers=None, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer()}"
    kwargs.setdefault("timeout", REQUEST_TIMEOUT_SECONDS)
    response = requests.request(method, url, headers=request_headers, **kwargs)
    if response.status_code == 401:
        try:
            response.close()
        except Exception:
            pass
        request_headers["Authorization"] = f"Bearer {get_active_bearer(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, **kwargs)
    return response


def get_dld_token(emirates_id):
    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": str(emirates_id),
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile",
    }
    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"
    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Content-Type": "application/json",
    }
    response = request_with_bearer("post", url, headers=headers)
    payload = response_payload(response)
    errors = api_errors(payload)
    if response.status_code >= 400 or errors:
        raise RuntimeError(f"DLD token failed for {emirates_id} HTTP {response.status_code}: {compact_payload(errors or payload)}")
    token = payload.get("token") if isinstance(payload, dict) else None
    if not token:
        raise RuntimeError(f"DLD token response has no token for {emirates_id}: {compact_payload(payload)}")
    return token


def dld_headers(dld_token):
    return {
        "accept": "text/plain",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
    }


def get_contract_history_result(dld_token):
    headers = dld_headers(dld_token)
    urls = [
        ("actual", f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"),
        ("proxy", f"{DLD_PROXY_URL}/ejariservices/properties/getcontracts"),
    ]
    attempts = []
    for source, url in urls:
        response = request_with_bearer("get", url, headers=headers)
        payload = response_payload(response)
        result = {
            "source": source,
            "url": url,
            "status_code": response.status_code,
            "response": payload,
            "headers": final_auth_headers(response, headers),
        }
        attempts.append(result)
        if response.status_code < 400 and not api_errors(payload):
            result["attempts"] = attempts
            return result
    result = attempts[-1] if attempts else {"status_code": None, "response": None}
    result["attempts"] = attempts
    return result


def extract_history_contracts(history_payload):
    response_data = history_payload.get("Response") if isinstance(history_payload, dict) else {}
    contracts = []
    for group, role in (("OwnerContracts", "owner"), ("TenantContracts", "tenant")):
        items = response_data.get(group) if isinstance(response_data, dict) else []
        for contract in items or []:
            if isinstance(contract, dict):
                item = dict(contract)
                item["_history_group"] = group
                item["_history_role"] = role
                contracts.append(item)
    return contracts


def get_contract_details_result(contract_row_value, dld_token):
    headers = dld_headers(dld_token)
    headers["Content-Type"] = "application/json-patch+json"
    urls = [
        ("actual", f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/{contract_row_value}"),
        ("proxy", f"{DLD_PROXY_URL}/ejariservices/contracts/{contract_row_value}"),
    ]
    attempts = []
    for source, url in urls:
        response = request_with_bearer("get", url, headers=headers)
        payload = response_payload(response)
        result = {
            "source": source,
            "url": url,
            "status_code": response.status_code,
            "response": payload,
            "headers": final_auth_headers(response, headers),
        }
        attempts.append(result)
        details = ((payload or {}).get("Response") or {}).get("ContractDetails") if isinstance(payload, dict) else None
        if response.status_code < 400 and not api_errors(payload) and isinstance(details, dict):
            result["attempts"] = attempts
            return result
    result = attempts[-1] if attempts else {"status_code": None, "response": None}
    result["attempts"] = attempts
    return result


In [ ]:

# @title Contract and auto-cancel helpers

def display_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("Value") or value.get("Name") or ""
    return value or ""


def first_present(*values):
    for value in values:
        if value not in (None, ""):
            return value
    return ""


def contract_row_value(contract):
    return first_present(
        contract.get("AssociatedContractRowValue"),
        contract.get("ContractRowValue"),
        contract.get("ContractDataRowValue"),
        contract.get("DataRowValue"),
    )


def contract_number(contract):
    return first_present(contract.get("ContractNumber"), contract.get("ContractNo"), contract.get("ContractId"))


def contract_status_text(contract):
    return str(display_name(contract.get("ContractStatus"))).strip()


def is_pending_contract(contract):
    return contract_status_text(contract).lower() == "pending"


def property_title(contract, details=None):
    title = display_name(contract.get("Title"))
    if title:
        return title
    props = (details or {}).get("AssociatedProperties") or []
    if not props:
        return ""
    prop = props[0] or {}
    unit = prop.get("UnitNumber")
    building = display_name(prop.get("Building"))
    area = display_name(prop.get("Area"))
    if unit and building and area:
        return f"Unit {unit} in {building} ({area})"
    if unit and area:
        return f"Unit {unit} ({area})"
    return first_present(unit, building, area)


def parse_datetime(value):
    if not value:
        return None
    if not isinstance(value, str):
        return None
    text = value.strip()
    for fmt in (
        "%Y-%m-%dT%H:%M:%S.%f%z",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
    ):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass
    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00"))
    except ValueError:
        return None


def format_datetime(value):
    dt = parse_datetime(value)
    return dt.strftime("%Y-%m-%d %H:%M:%S") if dt else ""


def add_days(value, days):
    dt = parse_datetime(value)
    return (dt + timedelta(days=days)).strftime("%Y-%m-%d %H:%M:%S") if dt else ""


def auto_cancel_base(details, contract):
    owner_signing_date = details.get("OwnerContractSigningDate")
    if owner_signing_date:
        return owner_signing_date, "OwnerContractSigningDate"
    return "", "missing OwnerContractSigningDate"


def build_report_row(emirates_id, contract, details, detail_result):
    base_date, base_source = auto_cancel_base(details, contract)
    return {
        "emirates_id": emirates_id,
        "history_role": contract.get("_history_role", ""),
        "history_group": contract.get("_history_group", ""),
        "contract_number": first_present(details.get("ContractNumber"), contract_number(contract)),
        "contract_row_value": first_present(details.get("DataRowValue"), contract_row_value(contract)),
        "property_id": contract.get("PropertyId", ""),
        "property_type": first_present(contract.get("PropertyType"), ((details.get("AssociatedProperties") or [{}])[0] or {}).get("PropertyTypeId")),
        "property_row_value": contract.get("RowValue", ""),
        "property_name": property_title(contract, details),
        "history_status": contract_status_text(contract),
        "detail_status": display_name(details.get("ContractStatus")),
        "workflow_status": display_name(details.get("ContractWorkflowStatus")),
        "is_owner_signed": details.get("IsContractSignedByOwner"),
        "is_tenant_signed": details.get("IsContractSignedByTenant"),
        "owner_contract_signing_date": format_datetime(details.get("OwnerContractSigningDate")),
        "tenant_contract_signing_date": format_datetime(details.get("TenantContractSigningDate")),
        "creation_date": format_datetime(first_present(details.get("CreationDate"), contract.get("ContractCreationDate"))),
        "autocancel_base_source": base_source,
        "autocancel_base_date": format_datetime(base_date),
        "auto_cancel_1_day_date": add_days(base_date, 1),
        "auto_cancel_5_day_date": add_days(base_date, 5),
        "contract_details_endpoint": detail_result.get("source", ""),
        "contract_details_url": detail_result.get("url", ""),
    }


def consolidate_pending_contracts(contracts):
    consolidated = {}
    missing_row_value = 0
    for contract in contracts:
        if not is_pending_contract(contract):
            continue
        row_value = contract_row_value(contract)
        if not row_value:
            missing_row_value += 1
            continue
        existing = consolidated.get(row_value)
        if existing is None:
            consolidated[row_value] = dict(contract)
            consolidated[row_value]["_source_groups"] = [contract.get("_history_group", "")]
            consolidated[row_value]["_roles"] = [contract.get("_history_role", "")]
            continue
        group = contract.get("_history_group", "")
        role = contract.get("_history_role", "")
        if group and group not in existing["_source_groups"]:
            existing["_source_groups"].append(group)
        if role and role not in existing["_roles"]:
            existing["_roles"].append(role)
    return list(consolidated.values()), missing_row_value


In [ ]:

# @title Output writers
REPORT_FIELDS = [
    "emirates_id",
    "history_role",
    "history_group",
    "contract_number",
    "contract_row_value",
    "property_id",
    "property_type",
    "property_row_value",
    "property_name",
    "history_status",
    "detail_status",
    "workflow_status",
    "is_owner_signed",
    "is_tenant_signed",
    "owner_contract_signing_date",
    "tenant_contract_signing_date",
    "creation_date",
    "autocancel_base_source",
    "autocancel_base_date",
    "auto_cancel_1_day_date",
    "auto_cancel_5_day_date",
    "contract_details_endpoint",
    "contract_details_url",
]

SUMMARY_FIELDS = [
    "emirates_id",
    "history_source",
    "history_total_rows",
    "pending_unique_contracts",
    "processed_count",
    "success_count",
    "failure_count",
    "contracts_missing_row_value",
    "elapsed_seconds",
    "error",
]

FAILURE_FIELDS = [
    "emirates_id",
    "contract_number",
    "contract_row_value",
    "property_id",
    "property_name",
    "stage",
    "http_status_code",
    "error",
    "url",
    "curl_path",
    "response_path",
]

report_rows = []
summary_rows = []
failure_rows = []


def write_outputs():
    write_csv(RUN_DIR / "pending_contract_autocancel_report.csv", report_rows, REPORT_FIELDS)
    write_json(RUN_DIR / "pending_contract_autocancel_report.json", report_rows)
    write_csv(RUN_DIR / "summary.csv", summary_rows, SUMMARY_FIELDS)
    write_json(RUN_DIR / "summary.json", summary_rows)
    write_csv(RUN_DIR / "failures.csv", failure_rows, FAILURE_FIELDS)
    write_json(RUN_DIR / "failures.json", failure_rows)


def save_history(emirates_id, history_result):
    path = HISTORY_DIR / f"history_{safe_filename(emirates_id)}.json"
    write_json(path, history_result)
    return str(path)


def save_detail(emirates_id, contract, detail_result, details):
    row_value = contract_row_value(contract)
    contract_no = first_present(details.get("ContractNumber"), contract_number(contract), row_value)
    path = DETAIL_DIR / f"details_{safe_filename(emirates_id)}_{safe_filename(contract_no)}_{safe_filename(row_value, 60)}.json"
    write_json(path, {
        "emirates_id": emirates_id,
        "history_contract": contract,
        "contract_details_api_result": detail_result,
        "contract_details": details,
    })
    return str(path)


def save_failure(emirates_id, contract, stage, result=None, error=None):
    result = result or {}
    row_value = contract_row_value(contract)
    prefix = f"{safe_filename(emirates_id)}_{safe_filename(contract_number(contract) or row_value)}_{safe_filename(stage)}"
    response_path = FAILURE_DIR / f"{prefix}_response.json"
    curl_path = FAILURE_DIR / f"{prefix}_request.sh"
    write_json(response_path, result)
    headers = result.get("headers") or {}
    method = result.get("method") or "GET"
    url = result.get("url") or ""
    curl_path.write_text(make_curl(method, url, headers), encoding="utf-8")
    row = {
        "emirates_id": emirates_id,
        "contract_number": contract_number(contract),
        "contract_row_value": row_value,
        "property_id": contract.get("PropertyId", ""),
        "property_name": property_title(contract),
        "stage": stage,
        "http_status_code": result.get("status_code", ""),
        "error": error or compact_payload(api_errors(result.get("response")) or result.get("response") or result.get("error") or "Unknown failure"),
        "url": url,
        "curl_path": str(curl_path),
        "response_path": str(response_path),
    }
    failure_rows.append(row)
    return row


In [ ]:

# @title Run pending contract auto-cancel audit
print("Run folder:", RUN_DIR)

for emirates_id in OWNER_EMIRATES_IDS:
    if master_selection != "all":
        decision = ask_choice(
            f"Process Emirates ID {emirates_id}?",
            [("yes", "Yes"), ("no", "No")],
            default="yes",
            title="Process Emirates ID",
        )
        if decision != "yes":
            print(f"\nSkipping Emirates ID {emirates_id}")
            continue

    print(f"\nProcessing Emirates ID {emirates_id}")
    started_at = time.time()
    processed_count = 0
    success_count = 0
    failure_count = 0
    missing_row_count = 0
    history_source = ""
    history_total_rows = 0
    pending_unique_count = 0
    error_text = ""

    try:
        CURRENT_BEARER_TOKEN = None
        get_active_bearer(force_refresh=True)
        dld_token = get_dld_token(emirates_id)

        history_result = get_contract_history_result(dld_token)
        history_source = history_result.get("source", "")
        save_history(emirates_id, history_result)
        if history_result.get("status_code") is None or history_result.get("status_code") >= 400 or api_errors(history_result.get("response")):
            raise RuntimeError(f"Contract history failed: {compact_payload(history_result)}")

        contracts = extract_history_contracts(history_result.get("response") or {})
        history_total_rows = len(contracts)
        pending_contracts, missing_row_count = consolidate_pending_contracts(contracts)
        pending_unique_count = len(pending_contracts)

        print(f"  contract history rows={history_total_rows}, pending unique={pending_unique_count}, missing row value={missing_row_count}")

        for contract in pending_contracts:
            processed_count += 1
            row_value = contract_row_value(contract)
            detail_result = get_contract_details_result(row_value, dld_token)
            payload = detail_result.get("response") or {}
            details = ((payload.get("Response") or {}).get("ContractDetails") if isinstance(payload, dict) else None) or {}

            if detail_result.get("status_code") is None or detail_result.get("status_code") >= 400 or api_errors(payload) or not details:
                failure_count += 1
                failure = save_failure(
                    emirates_id,
                    contract,
                    "contract_details",
                    detail_result,
                    error="Unable to load contract details",
                )
                print(f"  FAIL details | {contract_number(contract) or row_value} | {failure['response_path']}")
                continue

            save_detail(emirates_id, contract, detail_result, details)
            report_rows.append(build_report_row(emirates_id, contract, details, detail_result))
            success_count += 1

    except Exception as exc:
        error_text = str(exc)
        failure_count += 1
        print(f"  ERROR: {error_text}")
    finally:
        summary_rows.append({
            "emirates_id": emirates_id,
            "history_source": history_source,
            "history_total_rows": history_total_rows,
            "pending_unique_contracts": pending_unique_count,
            "processed_count": processed_count,
            "success_count": success_count,
            "failure_count": failure_count,
            "contracts_missing_row_value": missing_row_count,
            "elapsed_seconds": round(time.time() - started_at, 2),
            "error": error_text,
        })
        write_outputs()
        print(
            f"  totals: history={history_total_rows}, pending_unique={pending_unique_count}, "
            f"processed={processed_count}, success={success_count}, failure={failure_count}"
        )

print("\nDONE")
print("Run folder:", RUN_DIR)
print("Rows written:", len(report_rows))
print("Failures:", len(failure_rows))
print("Report CSV:", RUN_DIR / "pending_contract_autocancel_report.csv")


In [ ]:

# @title Preview report
try:
    import pandas as pd
    preview_columns = [
        "emirates_id",
        "contract_number",
        "property_id",
        "property_name",
        "is_owner_signed",
        "is_tenant_signed",
        "owner_contract_signing_date",
        "autocancel_base_source",
        "auto_cancel_1_day_date",
        "auto_cancel_5_day_date",
    ]
    display(pd.DataFrame(report_rows)[preview_columns])
except Exception:
    for row in report_rows:
        print(
            row["emirates_id"],
            row["contract_number"],
            row["property_id"],
            row["owner_contract_signing_date"],
            row["auto_cancel_1_day_date"],
            row["auto_cancel_5_day_date"],
        )
